# PyTorch Tensors & Autograd

## Introduction

Everything in this series — attention, gradient monitoring, LoRA, GRPO — runs on top of one mechanism: PyTorch's **autograd engine** for [automatic differentiation](/notebooks/deep/03.html). Without a precise mental model of `autograd`, gradient charts can be misread, bugs can silently corrupt training, and diagnosing loss explosions becomes difficult. This tutorial builds that model from scratch:

- What a tensor is at the memory level, and what `requires_grad=True` actually turns on
- How the computation graph is constructed during the forward pass — the nodes, the edges, the `grad_fn` chain
- What `.backward()` does: reverse-mode automatic differentiation, the chain rule as a graph traversal, gradient accumulation
- The four-stage training loop: forward → loss → backward → update
- Writing a two-layer network with no `nn.Module` — raw tensors, manual gradient zeroing, manual weight update — so nothing is hidden
- Rebuilding it with `nn.Module`, `nn.Linear`, and `optim.SGD` and understanding exactly what each abstraction is doing
- The five autograd bugs that will bite you: forgetting `.zero_grad()`, in-place ops, detaching at the wrong time, accumulating gradients across batches, and the `loss.backward()` + `retain_graph` trap

By the end, readers should be able to read a computation graph, predict which tensors have gradients, and understand what happens when `.backward()` is called.

---

## 1. Tensors at the Memory Level

A PyTorch `Tensor` is a view over a contiguous block of typed memory — a `Storage` — plus metadata: shape, stride, dtype, and device. The tesnors below share the same underlying storage, but with different stride interpretations (see below). This is why in-place operations on views can silently corrupt the graph: modifying `x` in-place affects `y`'s memory even though they appear to be different shapes:

In [1]:
import torch
from warnings import simplefilter
simplefilter("ignore", UserWarning)

x = torch.tensor([1.0, 2.0, 3.0])

# These ALL share memory:
y1 = x.view(3, 1)
y2 = x.reshape(3, 1)
y3 = x.T  # transpose
print(y1.storage().data_ptr() == x.storage().data_ptr())  # True
print(y2.storage().data_ptr() == x.storage().data_ptr())  # True
print(y3.storage().data_ptr() == x.storage().data_ptr())  # True

# This clearly does NOT share memory:
y4 = x.clone()
print(y4.storage().data_ptr() == x.storage().data_ptr())  # False

True
True
True
False


**Tensor stride.** This gives how many elements to skip in the underlying memory to move to the next position along a dimension. For the 1D case above, `stride=(1,)` means elements are contiguous — move 1 step in memory to get to the next element. For 2D tensors, stride has two values:

In [2]:
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0],
                  [7.0, 8.0, 9.0]])

y = x.T

print(x.stride())
print(y.stride())

# (3, 1) means:
#   - Skip 3 elements to move down one row
#   - Skip 1 element to move right one column

# (1, 3) means:
# when you transpose with `x.T` or use `.view`, it just reverses the stride.
# as shown above no new memory is allocated.

(3, 1)
(1, 3)


**The dtype matters for training.** Floating-point numbers work like [scientific notation in base 2]{.mark}: every value is stored as $\text{mantissa} \times 2^{\text{exponent}}$. The exponent sets the order of magnitude; the mantissa[^mantissa_decimal] provides the precision *within* that interval. Because the spacing between consecutive representable numbers scales with the exponent, the resolution is not uniform — near $1024$ the gap between adjacent values is around $10^{-4}$, while near $0.5$ the gap is around $10^{-7}$. Mental model: more significant digits, finer resolution.

Different dtypes allocate the 16 or 32 bits differently between exponent and mantissa:

| dtype | total bits | exponent bits | mantissa bits | decimal digits of precision | notes | recommended |
|---|---|---|---|---|---|---|
| `float32` | 32 | 8 | 23 | ~7 | standard training dtype | training from scratch |
| `float16` | 16 | 5 | 10 | ~3 | small exponent range — overflows above 65504 | older GPUs (V100) with loss scaling |
| `bfloat16` | 16 | 8 | 7 | ~2–3 | same exponent range as float32 — safe dynamic range | modern hardware (A100, H100, TPUs) |

Observe `float16` has only 5 exponent bits, which limits the range of representable values (up to ~65504). It also has only ~3 decimal digits of precision — enough to represent activations, but not enough to safely accumulate small gradient updates like $10^{-5}$, which can round to zero. `bfloat16` was designed to fix the range problem: it borrows bits from the mantissa to match `float32`'s 8-bit exponent. You give up a little precision, but you eliminate [overflow]{.underline} and [underflow]{.underline} during training. This tradeoff works well in practice because gradients don't need 7 digits of precision — they just need not to vanish. This will matter in [Tutorial 9]{.mark}. For now, we use `float32` throughout.

[^mantissa_decimal]: A mantissa of 23 is ~7 decimal places. To see this write: $\log_{10} 2^{23} = 23 \log_{10} 2 \approx 6.92.$

In [71]:
torch.set_printoptions(precision=8)

def fmt(val, dtype):
    v = torch.tensor(val, dtype=dtype).item()
    return f"{v:.8e}"

dtypes = [
    (" float32", torch.float32),        # high precision, large range
    ("bfloat16", torch.bfloat16),       # low  precision, large range
    (" float16", torch.float16)         # low  precision, small range
]

cases = [
    ("precision", 1/3), 
    ("underflow", 1e-8), 
    ("overflow",  1e5)
]

header = f"{'  dtype':<10s} " + "  ".join(f" {title+' ('+f'{val:.0e}'+')':<18s}" for title, val in cases)
print(header)
print("-" * len(header))
for label, dtype in dtypes:
    row = f"{label:<10s} " + "  ".join(f" {fmt(val, dtype):<18s}" for _, val in cases)
    print(row)

  dtype     precision (3e-01)    underflow (1e-08)    overflow (1e+05)  
------------------------------------------------------------------------
 float32    3.33333343e-01       9.99999994e-09       1.00000000e+05    
bfloat16    3.33984375e-01       1.00117177e-08       9.98400000e+04    
 float16    3.33251953e-01       0.00000000e+00       inf               


### `requires_grad`

Setting `requires_grad=True` tells PyTorch to track every operation on this tensor and record enough information to compute its gradient later:


In [74]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2          # y = x²
z = y + 3 * x       # z = x² + 3x

print(z)            # tensor(10., grad_fn=<AddBackward0>)
print(z.grad_fn)    # <AddBackward0 object>

z.backward()        # Compute gradients
print(x.grad)       # dz/dx = ∂z/∂y * dy/dx + ∂z / ∂x
                    #       = 1 * 2x + 3 = 7 OR as a function of x:
                    # dz/dx = 2x + 3     = 7 evaluated at x=2.

tensor(10., grad_fn=<AddBackward0>)
tensor(7.)


Notice `grad_fn`. This is the handle to the computation graph. The tensor `z` knows it was produced by an addition, and that addition node knows its inputs. This chain reaches all the way back to `x`.

[**Only leaf tensors accumulate gradients.**]{.underline} `x` is a *leaf* (you created it directly). `y` and `z` are *non-leaf* (computed from other tensors). After `.backward()`, only `x.grad` will be populated — not `y.grad` or `z.grad`. This is deliberate: storing gradients for every intermediate tensor in a deep network would be prohibitive. If you need a gradient for a non-leaf tensor, use `.retain_grad()` on it before the forward pass.[^leaf_retain]

[^leaf_retain]: This is exactly what `GradientMonitor` in [Tutorial 4]{.underline} avoids by using hooks instead.

---

## 2. The Computation Graph

During the forward pass, PyTorch builds a directed acyclic graph (DAG) where:

- **Nodes** are `Function` objects — one per operation (`MulBackward0`, `AddBackward0`, `PowBackward0`, etc.)
- **Edges** point from output to inputs — backwards through the computation
- **Leaf nodes** are tensors with `requires_grad=True` and no `grad_fn`


This graph exists entirely in Python/C++ memory. It is constructed lazily — only operations involving at least one `requires_grad=True` tensor create graph nodes. Operations on tensors with `requires_grad=False` are invisible to autograd.

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=False)  # b is invisible to autograd

c = a * b    # one graph node: MulBackward0
d = c ** 2   # one graph node: PowBackward0

print(d.grad_fn)          # <PowBackward0>
print(d.grad_fn.next_functions)
# ((<MulBackward0 object>, 0),)  — the graph edge to the multiply node

print(d.grad_fn.next_functions[0][0].next_functions)
# ((<AccumulateGrad object>, 0), (None, 0))
# AccumulateGrad is the leaf node for `a`; None is for `b` (not tracked)

You can walk the graph manually. This is worth doing once — it makes `.backward()` completely demystified:

In [ ]:
def print_graph(grad_fn, indent=0):
    if grad_fn is None:
        return
    print(" " * indent + str(grad_fn))
    for next_fn, _ in grad_fn.next_functions:
        print_graph(next_fn, indent + 2)

print_graph(d.grad_fn)
# PowBackward0
#   MulBackward0
#     AccumulateGrad   ← this is `a`

### The graph is rebuilt every forward pass

This is not TensorFlow 1.x — there is no *static graph* you define once. PyTorch rebuilds the computation graph fresh on every forward pass. This is why:

- You can use Python control flow (`if`, `for`, `while`) freely — the graph shape can change per call
- The graph is freed after `.backward()` unless you pass `retain_graph=True`
- You cannot call `.backward()` twice on the same graph without `retain_graph=True`

---

## 3. Reverse-Mode Automatic Differentiation

`.backward()` traverses the computation graph in *reverse topological order*, applying the chain rule at each node.

For a scalar output $L$ and a path $L \leftarrow z \leftarrow y \leftarrow x$, the chain rule gives:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial y} \cdot \frac{\partial y}{\partial x}$$

In the graph, this traversal is:

1. Start at the loss node with gradient $\frac{\partial L}{\partial L} = 1$
2. For each node, multiply the incoming gradient by the local Jacobian to get the gradient with respect to each input
3. Pass that gradient downstream to the input nodes
4. When a gradient reaches a leaf tensor (`AccumulateGrad`), add it to `.grad`

Let's verify this by hand:

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3        # y = x³,  dy/dx = 3x²
L = y * 4.0       # L = 4x³, dL/dy = 4

L.backward()

# By the chain rule: dL/dx = dL/dy * dy/dx = 4 * 3x² = 12x²
# At x=2: dL/dx = 12 * 4 = 48
print(x.grad)     # tensor(48.)

**Reverse-mode vs forward-mode.** *Reverse-mode AD* (what PyTorch uses) is efficient when you have many inputs and one scalar output — exactly the case for a loss function over millions of parameters. *Forward-mode AD* is efficient when you have one input and many outputs. For neural networks, reverse-mode wins decisively.

### Gradient accumulation

`.grad` is not overwritten — it is *accumulated* (added to). If you call `.backward()` twice without zeroing gradients between calls, you get the sum of the two gradients, not the second one alone:

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

(x ** 2).backward()
print(x.grad)   # tensor(4.)  — correct: d/dx x² = 2x = 4

(x ** 2).backward()
print(x.grad)   # tensor(8.)  — wrong! accumulated, not replaced

[This is why `optimizer.zero_grad()` exists]{.mark} — forgetting it is one of the most common training bugs. We will see the exact symptom — *gradient norms grow monotonically* across steps — in Tutorial 5.[^accum_bug]

[^accum_bug]: After $N$ steps without calling `.zero_grad()`, `.grad` holds the sum of $N$ gradient vectors. The effective update magnitude grows proportional to $N$ — a runaway that destabilizes training even at a reasonable learning rate.

---

## 4. The Training Loop from Scratch

Let's build a complete training loop with zero abstractions. The problem: fit a linear function $y = 3x + 2$ from noisy data.

In [ ]:
import torch

# --- Data ---
torch.manual_seed(42)
X = torch.randn(100, 1)           # 100 samples, 1 feature
Y = 3.0 * X + 2.0 + 0.1 * torch.randn(100, 1)   # y = 3x + 2 + noise

# --- Parameters (raw tensors, no nn.Module) ---
W = torch.randn(1, 1, requires_grad=True)   # weight
b = torch.zeros(1,    requires_grad=True)   # bias

lr = 0.1

for step in range(200):
    # --- Stage 1: Forward pass ---
    Y_pred = X @ W + b              # (100,1) @ (1,1) + (1,) → (100,1)

    # --- Stage 2: Loss ---
    loss = ((Y_pred - Y) ** 2).mean()   # mean squared error

    # --- Stage 3: Backward pass ---
    loss.backward()                 # populate W.grad and b.grad

    # --- Stage 4: Parameter update ---
    with torch.no_grad():           # don't track these ops
        W -= lr * W.grad
        b -= lr * b.grad

    # --- Zero gradients for next step ---
    W.grad.zero_()
    b.grad.zero_()

    if step % 40 == 0:
        print(f"step {step:3d}  loss={loss.item():.4f}  W={W.item():.3f}  b={b.item():.3f}")

Expected output (approximately):

```
step   0  loss=9.4231  W=0.712  b=0.097
step  40  loss=0.0132  W=2.891  b=1.943
step  80  loss=0.0104  W=2.987  b=1.987
step 120  loss=0.0101  W=2.999  b=1.998
step 160  loss=0.0100  W=3.001  b=2.000
```

Four things to notice:

**`torch.no_grad()` around the update.** The update `W -= lr * W.grad` is itself a tensor operation. Without `no_grad()`, PyTorch would try to add this to the computation graph, creating a new node that wraps `W` — making it no longer a leaf tensor, which means it can no longer accumulate gradients. `no_grad()` **disables graph construction** for the operations inside the context manager.

**`W.grad.zero_()` after the update, not before.** The convention is to zero gradients at the end of each step (or at the start of the next). If you zero before `loss.backward()` you get correct results; if you zero after the update you also get correct results. What breaks is forgetting to zero at all.

**In-place ops on the parameters.** `W -= lr * W.grad` uses *in-place subtraction*. This is safe here because it's inside `no_grad()` and `W` is a leaf. In-place ops on non-leaf tensors inside the computation graph will raise a `RuntimeError` about a tensor being modified in-place that is required for gradient computation — this is PyTorch protecting the graph from corruption.

**`.item()` for logging.** Calling `.item()` extracts a scalar from a single-element tensor as a Python float. This detaches it from the graph and is the correct way to log loss values. Logging `loss` directly keeps the tensor alive, which keeps the entire computation graph alive, which is a memory leak.

---

## 5. Rebuilding with `nn.Module` and `optim`

The raw-tensor version above is exactly what `nn.Module` and `optim.SGD` are doing. Let's rebuild it so you can see the correspondence:

In [ ]:
import torch
import torch.nn as nn

# --- Data (same as before) ---
torch.manual_seed(42)
X = torch.randn(100, 1)
Y = 3.0 * X + 2.0 + 0.1 * torch.randn(100, 1)

# --- Model ---
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=1, out_features=1)
        # nn.Linear creates W (weight) and b (bias) automatically
        # W is initialized with kaiming_uniform_, b with uniform_

    def forward(self, x):
        return self.linear(x)   # x @ W.T + b

model = LinearModel()

# --- Optimizer ---
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
# model.parameters() returns an iterator over all tensors with requires_grad=True
# SGD stores references to these tensors and updates them in-place

# --- Loss function ---
loss_fn = nn.MSELoss()

# --- Training loop ---
for step in range(200):
    # Stage 1: Forward
    Y_pred = model(X)

    # Stage 2: Loss
    loss = loss_fn(Y_pred, Y)

    # Stage 3: Backward
    optimizer.zero_grad()   # zero all parameter gradients
    loss.backward()         # populate all .grad tensors

    # Stage 4: Update
    optimizer.step()        # W -= lr * W.grad  for each parameter

    if step % 40 == 0:
        W = model.linear.weight.item()
        b = model.linear.bias.item()
        print(f"step {step:3d}  loss={loss.item():.4f}  W={W:.3f}  b={b:.3f}")

The correspondence is exact:

| Raw tensor version | `nn.Module` version |
|---|---|
| `W = torch.randn(..., requires_grad=True)` | `nn.Linear` creates `self.weight` |
| `Y_pred = X @ W + b` | `model(X)` calls `forward()` |
| `loss.backward()` | `loss.backward()` — identical |
| `W -= lr * W.grad` | `optimizer.step()` |
| `W.grad.zero_()` | `optimizer.zero_grad()` |

`nn.Module` adds three things the raw version lacks: parameter registration (so `model.parameters()` works), device management (`.to(device)` moves all parameters at once), and state serialization (`model.state_dict()` / `model.load_state_dict()`). It does not add any new mathematics.

### `optimizer.zero_grad()` placement

There is a subtle timing choice: zero before backward, or zero after update?

In [ ]:
# Style A — zero before backward (most common)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Style B — zero after update
loss.backward()
optimizer.step()
optimizer.zero_grad()

Both are correct for standard training. Style B enables gradient accumulation across multiple forward passes before a single update — useful for *large effective batch sizes* on limited GPU memory. We will use this in Tutorial 8 and Tutorial 9.

---

## 6. A Two-Layer Network

Linear regression is too simple to surface the interesting behaviors. Let's build a two-layer network to see how the gradient flows through multiple layers:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- Toy classification: XOR problem ---
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
Y = torch.tensor([0., 1., 1., 0.])   # XOR labels

# --- Raw tensor version: two-layer net ---
# Layer 1: (2 → 4) with ReLU
W1 = torch.randn(2, 4, requires_grad=True) * 0.5
b1 = torch.zeros(4,    requires_grad=True)

# Layer 2: (4 → 1) linear
W2 = torch.randn(4, 1, requires_grad=True) * 0.5
b2 = torch.zeros(1,    requires_grad=True)

optimizer = torch.optim.SGD([W1, b1, W2, b2], lr=0.5)

for step in range(2000):
    # Forward
    h  = F.relu(X @ W1 + b1)    # (4,4) — hidden layer
    out = (h @ W2 + b2).squeeze()  # (4,)  — output logit

    # Binary cross-entropy loss
    loss = F.binary_cross_entropy_with_logits(out, Y)

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        acc = ((out.detach().sigmoid() > 0.5).float() == Y).float().mean()
        print(f"step {step:4d}  loss={loss.item():.4f}  acc={acc.item():.2f}")

# Final check
with torch.no_grad():
    h   = F.relu(X @ W1 + b1)
    out = (h @ W2 + b2).squeeze().sigmoid()
    print("\nPredictions:", out.round())
    print("Targets:    ", Y)

Expected output:

```
step    0  loss=0.7891  acc=0.50
step  500  loss=0.4823  acc=0.75
step 1000  loss=0.2103  acc=1.00
step 1500  loss=0.1204  acc=1.00

Predictions: tensor([0., 1., 1., 0.])
Targets:     tensor([0., 1., 1., 0.])
```

Now let's inspect the *gradient magnitudes* to see what \"healthy\" looks like — this can serve as a baseline when something goes wrong:

In [ ]:
# After one backward pass, inspect gradient norms
optimizer.zero_grad()
h   = F.relu(X @ W1 + b1)
out = (h @ W2 + b2).squeeze()
loss = F.binary_cross_entropy_with_logits(out, Y)
loss.backward()

print(f"W1 grad norm: {W1.grad.norm().item():.4f}")
print(f"W2 grad norm: {W2.grad.norm().item():.4f}")
print(f"W1 weight norm: {W1.norm().item():.4f}")
print(f"W2 weight norm: {W2.norm().item():.4f}")
print(f"grad/weight ratio W1: {(W1.grad.norm() / W1.norm()).item():.4f}")
print(f"grad/weight ratio W2: {(W2.grad.norm() / W2.norm()).item():.4f}")

The gradient-to-weight ratio — $\|\nabla W\| / \|W\|$ — is a key diagnostic. A healthy value is roughly in the range $[10^{-3}, 10^{-1}]$. Too small means the weights are barely moving (vanishing gradients or too-small LR). Too large means the update is destabilizing the network (exploding gradients or too-large LR). We will build a proper monitor for this in Tutorial 4.

---

## 7. The Five Autograd Bugs

These are common autograd bugs. Each one has a specific symptom.

### Bug 1: Forgetting `.zero_grad()`

**Symptom:** Loss oscillates or diverges even with a correct model; *gradient norms grow monotonically* across steps.

In [ ]:
for step in range(100):
    loss = loss_fn(model(X), Y)
    loss.backward()
    optimizer.step()
    # MISSING: optimizer.zero_grad()
    # Gradients accumulate: grad_step_N = sum of grads from steps 1..N
    # Effective learning rate grows with step count

**Fix:** Always call `optimizer.zero_grad()` once per step. The canonical position is immediately before `loss.backward()`.

### Bug 2: In-place operation on a tensor needed for gradient computation

**Symptom:** `RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation`.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x * 2      # y depends on x

x.add_(1.0)    # in-place modification of x — corrupts the graph
               # PyTorch recorded the value of x at the time of `x * 2`
               # but x has now changed; the gradient would be wrong

y.sum().backward()   # RuntimeError

**Fix:** Use *out-of-place operations* (`x = x + 1.0`) on any tensor that is part of the computation graph. In-place ops are safe only on tensors that are not part of any graph (e.g., parameter updates inside `torch.no_grad()`).

### Bug 3: Detaching too early

**Symptom:** Loss trains fine, but a portion of the network never updates — its gradients are always zero.

In [ ]:
# Intended: train both encoder and classifier together
features = encoder(x)
features = features.detach()    # WRONG — cuts the graph here
logits   = classifier(features)
loss     = loss_fn(logits, y)
loss.backward()
# encoder.parameters() get zero gradients — they're upstream of the detach

`.detach()` creates a new tensor that shares storage with the original but has no `grad_fn`. The **backward pass cannot cross a `.detach()` boundary**. This is intentional when you want to *freeze* part of a network (e.g., a frozen backbone), but catastrophic if accidental.

**Fix:** Only detach tensors you explicitly want to freeze. Use `model.requires_grad_(False)` to freeze entire modules instead — it's explicit and harder to misplace.

### Bug 4: Calling `.backward()` twice without `retain_graph=True`

**Symptom:** `RuntimeError: Trying to backward through the graph a second time`.

```python
loss = loss_fn(model(x), y)
loss.backward()
loss.backward()   # RuntimeError — graph has been freed
```

PyTorch **frees the computation graph** after the first `.backward()` to reclaim memory. If you need to call `.backward()` twice (e.g., computing second-order gradients, or two different loss terms on the same graph), use `loss.backward(retain_graph=True)` on all calls except the last.

In practice, if you find yourself doing this, you almost certainly want to sum your loss terms first and call `.backward()` once: `(loss_a + loss_b).backward()`.

### Bug 5: Gradients of non-leaf tensors not retained

**Symptom:** Inspecting `some_intermediate.grad` returns `None` even after `.backward()`.

In [ ]:
x  = torch.randn(3, requires_grad=True)
h  = torch.relu(x * 2)     # non-leaf
out = h.sum()

out.backward()

print(x.grad)   # tensor([...]) — correct, x is a leaf
print(h.grad)   # None — h is non-leaf, gradient not retained by default

**Fix:** Call `h.retain_grad()` before the forward pass if you need to inspect intermediate gradients. This is exactly what the `GradientMonitor` in Tutorial 4 does under the hood — it *hooks into the backward pass* to capture gradients without needing to call `retain_grad()` on every tensor.

---

## 8. Weight Initialization

Before Part II, one more concept that affects everything downstream: *weight initialization*.

If you initialize all weights to zero, every neuron in a layer computes the same function, receives the same gradient, and updates identically. The layer stays symmetric forever — you've wasted every parameter after the first in each layer. This is the **symmetry-breaking** problem.

If you initialize weights too large, activations saturate (for sigmoid/tanh) or explode (for ReLU). The loss diverges or the gradient vanishes in the saturation region.

The standard solutions:

**Xavier / Glorot initialization** (for tanh/sigmoid activations): samples weights from:

$$W \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}},\ \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

The derivation: choose the scale so that the *variance of the output equals the variance of the input* — $\text{Var}(y) = \text{Var}(x)$ — under the assumption that activations are approximately linear. This keeps signal magnitude stable across layers.

**Kaiming / He initialization** (for ReLU activations): samples from:

$$W \sim \mathcal{N}\!\left(0,\ \sqrt{\frac{2}{n_{\text{in}}}}\right)$$

The factor of 2 accounts for the fact that ReLU zeroes half its inputs on average, halving the effective variance. Without the 2, the signal shrinks by $\sqrt{2}$ per layer.

In [ ]:
import torch.nn as nn

layer = nn.Linear(256, 256)

# Xavier (default for nn.Linear)
nn.init.xavier_uniform_(layer.weight)

# Kaiming (recommended for ReLU networks)
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')

# What nn.Linear does by default: kaiming_uniform_ with mode='fan_in'
# Which is also reasonable for ReLU — just the uniform variant

`nn.Linear` uses Kaiming uniform by default. You'll rarely need to override this for standard architectures. Where initialization matters critically is in the **Transformer**: the *residual stream* means many layers add to the same signal, and the scale of each layer's output contribution needs to account for that — a topic we address in Tutorial 2.

---

## 9. Putting It Together: A Training Loop Template

Here is the complete template we will use throughout this series — every part labeled:

In [ ]:
import torch
import torch.nn as nn

def train(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_fn,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    max_steps: int,
    grad_clip: float = 1.0,
):
    model.train()
    step = 0

    for batch in dataloader:
        if step >= max_steps:
            break

        # --- Move data to device ---
        x, y = batch
        x, y = x.to(device), y.to(device)

        # --- Forward pass ---
        logits = model(x)

        # --- Loss ---
        loss = loss_fn(logits, y)

        # --- Backward pass ---
        optimizer.zero_grad()
        loss.backward()

        # --- Gradient clipping (optional but standard) ---
        # Clips the global gradient norm to `grad_clip`
        # Prevents exploding gradients from destabilizing training
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        # --- Parameter update ---
        optimizer.step()

        # --- Logging (detached from graph) ---
        if step % 100 == 0:
            print(f"step {step:5d}  loss={loss.item():.4f}  grad_norm={grad_norm:.4f}")

        step += 1

---

## 10. Custom Autograd Functions

Every operation above uses PyTorch's built-in backward rules. Three situations require defining your own:

- **Non-differentiable ops.** `round()`, `sign()`, `argmax()` have zero or undefined gradients almost everywhere. To train *through* these operations you provide a *surrogate gradient* — mathematically wrong but empirically effective.
- **Numerically unstable built-in gradient.** The exact derivative can overflow or underflow in float32. A custom backward lets you implement the stable version directly.
- **Custom CUDA kernels.** When writing a fused GPU kernel you supply the backward pass manually.

The mechanism is `torch.autograd.Function` — subclass it and implement two static methods:

```python
class MyFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)   # stash tensors needed in backward
        return output

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors     # retrieve stashed tensors
        return grad_output * local_gradient
```

Call it with `MyFunction.apply(x)` — **never** instantiate directly. The `ctx` object is the bridge between the two methods: anything saved with `ctx.save_for_backward(...)` in `forward` is available via `ctx.saved_tensors` in `backward`.

In [75]:
import torch

class ClampedExpFunction(torch.autograd.Function):
    """
    Forward:  exp(x) clamped to [0, 1e6] — prevents overflow for large x.
    Backward: uses the saved (clamped) output instead of recomputing exp(x),
              so the backward pass is also overflow-safe.
    This is a deliberate approximation: the gradient near the saturation
    ceiling is wrong, but the forward pass never blows up.
    """
    @staticmethod
    def forward(ctx, x):
        out = x.exp().clamp(max=1e6)
        ctx.save_for_backward(out)      # save output, not input
        return out

    @staticmethod
    def backward(ctx, grad_output):
        out, = ctx.saved_tensors        # clamped exp(x)
        return grad_output * out        # chain rule: dL/dx = dL/dout * exp(x)


# --- Quick sanity check ---
x = torch.tensor([0.5, 1.0, 2.0], requires_grad=True)
y = ClampedExpFunction.apply(x)
print("y      :", y)                    # [exp(0.5), exp(1.0), exp(2.0)]
y.sum().backward()
print("x.grad :", x.grad)              # same — gradient = exp(x)

# Large input: forward is safe (clamped), backward uses the saved clamped value
x_large = torch.tensor([100.0], requires_grad=True)
y_large = ClampedExpFunction.apply(x_large)
print("\nforward (clamped):", y_large)      # 1e6, not inf
y_large.backward()
print("backward (safe):  ", x_large.grad)  # 1e6, not nan

y      : tensor([1.64872122, 2.71828175, 7.38905621],
       grad_fn=<ClampedExpFunctionBackward>)
x.grad : tensor([1.64872122, 2.71828175, 7.38905621])

forward (clamped): tensor([1000000.], grad_fn=<ClampedExpFunctionBackward>)
backward (safe):   tensor([1000000.])


### Verifying with `gradcheck`

`torch.autograd.gradcheck` numerically approximates the Jacobian via *finite differences*:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + \epsilon e_i) - f(x - \epsilon e_i)}{2\epsilon}$$

and compares it to the analytical Jacobian produced by your `backward`. If they agree within tolerance, your implementation is correct. **Always run this when writing a custom `Function`** — it immediately catches sign errors, index mistakes, and missing chain-rule terms.

Two requirements: use `float64` inputs (finite differences need high precision) and stay away from inputs that activate any intentional approximation (e.g., the clamping region here).

In [76]:
x_check = torch.tensor([0.1, 0.5, 1.0], dtype=torch.float64, requires_grad=True)
result = torch.autograd.gradcheck(ClampedExpFunction.apply, x_check)
print("gradcheck passed:", result)   # True

gradcheck passed: True


### The Straight-Through Estimator (STE)

`round()` has gradient zero almost everywhere — its output is locally flat, so `loss.backward()` delivers no learning signal to anything upstream of a rounding layer.

The **straight-through estimator** is a pragmatic fix: round normally in the forward pass, but let gradients flow *as if no rounding happened* in the backward pass. This is mathematically wrong but empirically effective, and it is the standard technique in **quantization-aware training (QAT)** for deploying models on integer hardware.

In [77]:
class StraightThroughRound(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.round()       # discrete — zero gradient everywhere

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output     # pretend rounding didn't happen

st_round = StraightThroughRound.apply

# --- Toy demo: learn a value whose rounded form matches a target ---
torch.manual_seed(0)
target = torch.tensor([3.0])
x = torch.tensor([0.0], requires_grad=True)

opt = torch.optim.SGD([x], lr=0.1)

for step in range(60):
    loss = (st_round(x) - target) ** 2
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 12 == 0:
        print(f"step {step:3d}  x={x.item():.4f}  rounded={st_round(x.detach()).item():.1f}  loss={loss.item():.4f}")

# Compare: standard rounding has zero gradient — learning is impossible
x_bad = torch.tensor([0.0], requires_grad=True)
loss_bad = (x_bad.round() - target) ** 2
loss_bad.backward()
print(f"\nwithout STE → x.grad = {x_bad.grad.item()}")   # 0.0 — no learning signal

step   0  x=0.6000  rounded=1.0  loss=9.0000
step  12  x=2.6000  rounded=3.0  loss=0.0000
step  24  x=2.6000  rounded=3.0  loss=0.0000
step  36  x=2.6000  rounded=3.0  loss=0.0000
step  48  x=2.6000  rounded=3.0  loss=0.0000

without STE → x.grad = 0.0


### Wrapping in `nn.Module`

Wrap the `Function` in a thin `nn.Module` so it composes cleanly in `nn.Sequential` and moves with `.to(device)`:

```python
class ClampedExp(nn.Module):
    def forward(self, x):
        return ClampedExpFunction.apply(x)

class QuantizeRound(nn.Module):
    def forward(self, x):
        return StraightThroughRound.apply(x)
```

The module itself holds no parameters — it is purely an adapter. If the custom op *has* learnable parameters (e.g., a parameterized quantization scale), define them in the `Module` and pass them as extra tensor arguments to `Function.forward`; stash them with `ctx.save_for_backward` if needed in `backward`.

Three additions beyond the minimal version:

**`model.train()`** sets the model to *training mode*. This affects `nn.Dropout` (which zeroes activations randomly during training but is a no-op at eval time) and `nn.BatchNorm` (which uses batch statistics during training and running statistics at eval). Always pair it with `model.eval()` during validation.

**`clip_grad_norm_`** rescales the gradient if its **global L2 norm** exceeds `grad_clip`. It returns the norm before clipping — log this. Seeing `grad_norm` consistently near `grad_clip` is a warning sign that your model is on the edge of instability. We instrument this properly in Tutorial 5.

**`loss.item()`** extracts the loss as a Python float and *detaches it from the graph*. [Never log the raw `loss` tensor]{.underline} — it keeps the entire computation graph alive until the next call to `.backward()`.

---

## Summary

| Concept | Key detail |
|---|---|
| Tensor storage | Tensors are views over a `Storage`. Views share memory — in-place ops affect all views. |
| `requires_grad=True` | Turns on computation graph tracking for this tensor and all tensors derived from it. |
| `grad_fn` | Handle to the graph node that produced this tensor. `None` for leaf tensors. |
| Leaf tensors | Tensors created directly (not computed). Only leaves accumulate `.grad` after `.backward()`. |
| Graph construction | Built dynamically during the forward pass. Freed after `.backward()` unless `retain_graph=True`. |
| Reverse-mode AD | Traverses the graph backward, applies chain rule at each node, accumulates at leaves. |
| Gradient accumulation | `.grad` is *added to*, not replaced. Must call `.zero_grad()` before each backward pass. |
| `torch.no_grad()` | Disables graph construction inside the context. Required for parameter updates. |
| `.item()` | Extracts a Python scalar, detaches from graph. Use for all logging. |
| In-place ops | Safe inside `no_grad()` on leaves. Illegal on non-leaf tensors in the graph. |
| `.detach()` | Creates a graph-disconnected view. Intentional for freezing; catastrophic if accidental. |
| Xavier init | For tanh/sigmoid: scales by $\sqrt{6/(n_\text{in}+n_\text{out})}$. Preserves variance. |
| Kaiming init | For ReLU: scales by $\sqrt{2/n_\text{in}}$. The factor-2 corrects for ReLU's half-zeroing. |
| `grad_clip` | Global gradient norm clipping. `clip_grad_norm_` returns the pre-clip norm — log it. |
| `model.train()` | Activates dropout and batch norm training mode. Pair with `model.eval()` at validation. |
| `torch.autograd.Function` | Subclass with static `forward(ctx, ...)` and `backward(ctx, ...)`. Defines a custom op with a custom gradient. Call via `.apply()`, never instantiate directly. |
| `ctx.save_for_backward()` | Stash tensors in `forward` to retrieve in `backward` via `ctx.saved_tensors`. Only tensors needed for gradient computation should be saved here. |
| `gradcheck` | Numerically validates a custom `backward` via finite differences. Always run on `float64` inputs and away from any intentional gradient approximations. |

---

## Exercises

These are not optional polish — they are the tutorial. Reading without doing them will leave gaps.

**1.** Walk the computation graph of `z = (x**2 + y).exp()` manually using `.grad_fn.next_functions`. Draw the DAG on paper before running the code.

**2.** Reproduce the "forgetting `.zero_grad()`" bug deliberately. Plot `W.grad.norm()` vs step number and observe the monotonic growth.

**3.** Trigger the in-place bug: build a graph, modify a tensor in-place, call `.backward()`, and read the error message carefully. Understand which tensor it refers to.

**4.** Add `retain_grad()` to the hidden layer `h` in the two-layer network and inspect its gradient after `.backward()`. Verify the chain rule: $\frac{\partial L}{\partial h} = \frac{\partial L}{\partial \hat{y}} \cdot W_2^\top$.

**5.** Implement Xavier initialization from the formula by hand (do not use `nn.init.xavier_uniform_`). Train the two-layer XOR network with your initialization vs random normal and compare convergence speed.

**6.** Modify the training loop template to accumulate gradients over 4 mini-batches before each optimizer step. Verify that it produces the same loss curve as a single batch of 4x the size.

**7.** Implement a custom `Swish` activation ($f(x) = x \cdot \sigma(x)$, where $\sigma$ is sigmoid) using `torch.autograd.Function` with the hand-derived backward ($f'(x) = \sigma(x) + x \cdot \sigma(x)(1 - \sigma(x))$). Verify correctness with `gradcheck`. Then train the two-layer XOR network from Section 6 using `Swish` in place of `ReLU` and compare convergence speed.